<a href="https://colab.research.google.com/github/Aryan2079/LLM_finetuning/blob/main/non_instruction_pretrain_llm_finetuning_on_domain_specific_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U peft transformers bitsandbytes accelerate

In [ ]:
!pip install --upgrade transformers


In [ ]:
!pip install -U trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 13.8 MB/s eta 0:00:00


In [ ]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 78.3 MB/s eta 0:00:00


**Non-Instruction finetuning dataset**


*   hugginfcefw/fineweb
*   ncbi/pubmed
*   datajuicer/the-ple-pubmed-abstracts-refined-by-data-juicer
*   open-llm-leaderboard/open_llm_corpur
*   skylion007/openwebtext
*   armanc/scientific_paper


**instruction finetuning data**
*    https://huggingface.co/datasets/Amod/mental_health_counseling_conversations
*    https://huggingface.co/datasets/yahma/alpaca-cleaned
*    https://huggingface.co/datasets/Open-Orca/OpenOrca
*    https://huggingface.co/datasets/tatsu-lab/alpaca
*    https://huggingface.co/datasets/OpenAssistant/oasst1

**preference alignment dataset**
*   https://huggingface.co/datasets/Anthropic/hh-rlhf
*   https://huggingface.co/datasets/argilla/ultrafeedback-binarized-preferences-cleaned
*   https://huggingface.co/datasets/xinlai/Math-Step-DPO-10K






**Prebuilt data from hugginface data hub**

In [ ]:
from datasets import load_dataset, Dataset

In [ ]:
dataset = load_dataset("roneneldan/TinyStories",split="train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [ ]:
print(dataset)
print(len(dataset[0]['text']))

Dataset({
    features: ['text'],
    num_rows: 2119719
})
701


**our own custom data (non instruction data) for domain specific finetuning**

In [ ]:
import fitz


In [ ]:
def extract_text_from_pdf(pdf_path):
  text_blocks = []
  with fitz.open(pdf_path) as doc:
    for page in doc:
      text = page.get_text("text").strip()
      if text:
        text_blocks.append(text)
  return text_blocks

In [ ]:
pdf_text = extract_text_from_pdf("/content/Metformin.pdf")

In [ ]:
pdf_text

['Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis. \n \nClinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA red

In [ ]:
import re

def split_paragraph(page_text):
  paragraphs = []
  chunks = re.split(r'\n\s*\n', page_text)
  for chunk in chunks:
    clean = chunk.strip()
    if len(clean) > 30:
      paragraphs.append(clean)
  return paragraphs

In [ ]:
paragraphs = split_paragraph(*pdf_text)
paragraphs

['Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis.',
 'Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA redu

In [ ]:
paragraphs = [{'text': paragraph} for paragraph in paragraphs]
paragraphs

[{'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis.'},
 {'text': 'Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits h

In [ ]:
hf_dataset = Dataset.from_list(paragraphs)
hf_dataset

Dataset({
    features: ['text'],
    num_rows: 4
})

**lets select the model**

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling, BitsAndBytesConfig

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
tokenizer.eos_token

'</s>'

In [ ]:
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize_fn(example):
  tokens = tokenizer(example["text"], truncation=True, max_length=512, padding="max_length")
  tokens['labels'] = tokens['input_ids'].copy()
  return tokens

In [ ]:
tokenized = hf_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4
})

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name)

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

In [ ]:

prompt = "Clinical trails demonstrated that combining Atorvastating with Ezetimibe"

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt")

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0],skip_special_tokens=True))


Model Output:

Clinical trails demonstrated that combining Atorvastating with Ezetimibe improved the Atherosclerotic Vascular Events (AAVE) in patients with Type 2 Diabetes and Coronary Heart Disease.
In India, the market for Diabetic Cardiovascular Diseases is expected to register a CAGR of ~15% during the forecast period. This can be attributed to factors such as the growing prevalence of diabetes, increasing incidence of cardiovascular diseases and


In [ ]:
training_args = TrainingArguments(
    output_dir = "./llama-pharma-domain", # where we are going to save the model
    # overwrite_output_dir=True, # this will overwrite the previous content in the output dir
    num_train_epochs=2, # training epochs
    per_device_train_batch_size=2, # how many batches are we passing in one single go
    save_steps=500, # after how many steps are we saving the model ie. checkpoints
    save_total_limit=2, # how many steps to save. ex - last 2 checkpoints will be saved others will be removed as new checkpoints come
    logging_steps=50, # how many steps after to log
    learning_rate=2e-5, # learning rate
    fp16=True, # which precision are we going to load the model
    report_to="none" # if we want to keep out metrics somewhere like tensorboard
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=hf_dataset
)

if we just trained this directly then we would be fine-tuning the entire model. so we would get a memory overflow error and it would crash.

so we should use peft

In [ ]:
trainer.train()

ValueError: No columns in the dataset match the model's forward method signature: (input_ids, attention_mask, position_ids, past_key_values, inputs_embeds, labels, use_cache, cache_position, logits_to_keep, kwargs, label_ids, labels, label). The following columns have been ignored: [text]. Please check the dataset and model. You may need to set `remove_unused_columns=False` in `TrainingArguments`.

Now lets see the LoRA based method

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [ ]:
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [ ]:
tokenized = hf_dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto", # Ensures GPU usage
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none"
)

In [ ]:
q_lora_model = get_peft_model(model, lora_config)

In [ ]:
args = TrainingArguments(
    output_dir="./tinyllama-lora",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=q_lora_model,
    args=args,
    train_dataset=tokenized
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss


TrainOutput(global_step=5, training_loss=9.494598388671875, metrics={'train_runtime': 18.6525, 'train_samples_per_second': 1.072, 'train_steps_per_second': 0.268, 'total_flos': 63629646888960.0, 'train_loss': 9.494598388671875, 'epoch': 5.0})

In [ ]:
model_path = "/content/tinyllama-lora/checkpoint-5"

In [ ]:
trained_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [ ]:
prompt = "Clinical trails demonstrated that combining Atorvastating with Ezetimibe"

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = trained_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0],skip_special_tokens=True))


Model Output:

Clinical trails demonstrated that combining Atorvastating with Ezetimibe and Ramipril in patients with elevated LDL cholesterol levels lowered LDL-C by 30%, without increasing Atherogenic Index (AI) significantly.
Ezetimibe and atorvastatin are two HMG CoA reductase inhibitors. Both of them are the major metabolites produced from simvastatin. They are known to be absorbed by the gut wall and then enter the portal vein to the
